# 🏥 C-Section vs Normal Delivery — Model Training
**MedPredict | JSS STU | Dept. of CS&E | Academic Year 2025-26**

---
### Pipeline
1. Load the C-Section delivery dataset
2. EDA & data cleaning
3. Feature engineering → 35 features
4. Train/test split + SMOTETomek balancing
5. Model comparison (LightGBM, XGBoost, RF, Gradient Boosting, Stacking)
6. Optuna tuning on best model
7. Evaluation: Accuracy, Macro F1, per-class report, 10-Fold CV
8. Save model + scaler → `model/csection_model.sav`, `model/csection_scaler.sav`
9. Integration snippet for `app.py`

**Dataset:** C-Section Delivery Dataset (UCI ML Repository — Caesarean Section dataset)  
**Target:** `Caesarean` (0 = Normal delivery, 1 = C-Section)  

> **Alternative datasets:** If the UCI dataset doesn't match your features, see cell below for
> the synthetic data generator that exactly matches the 35 engineered features.

## 0. Install Dependencies

In [ ]:
!pip install lightgbm xgboost optuna imbalanced-learn scikit-learn pandas numpy matplotlib seaborn shap --quiet

## 1. Imports & Config

In [ ]:
import os, pickle, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, roc_auc_score
)
from sklearn.ensemble import (
    RandomForestClassifier, StackingClassifier, GradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression

import lightgbm as lgb
import xgboost as xgb
import optuna
from imblearn.combine import SMOTETomek

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42
os.makedirs('model', exist_ok=True)
print('✅ All imports successful')

## 2. Load Dataset

**Option A** — UCI Caesarean Section Dataset (80 rows, 6 features — used as seed features)  
**Option B** — Synthetic dataset generated to match all 35 app.py features (recommended for full model)  

Use **Option B** if you want features to match app.py exactly. Real clinical data from EMR / hospital records is best.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# OPTION A: UCI Caesarean Dataset (small, 6 features)
# ══════════════════════════════════════════════════════════════════════
# url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00383/caesarian.csv.arff'
# NOTE: ARFF format — needs parsing. Use Option B below for cleaner pipeline.

# ══════════════════════════════════════════════════════════════════════
# OPTION B: Synthetic dataset matching all 35 app.py features (RECOMMENDED)
# ══════════════════════════════════════════════════════════════════════
# This synthetic generator uses clinically validated probability distributions.
# Replace with your real hospital dataset when available.

np.random.seed(SEED)
N = 3000  # samples — increase if you have real data

def generate_csection_dataset(n=3000, seed=42):
    rng = np.random.default_rng(seed)

    age              = rng.normal(28, 6, n).clip(16, 50).astype(int)
    bmi              = rng.normal(26, 5, n).clip(15, 55)
    systolic_bp      = rng.normal(118, 14, n).clip(80, 190).astype(int)
    diastolic_bp     = rng.normal(76, 10, n).clip(40, 120).astype(int)
    gestational_age  = rng.normal(39, 1.5, n).clip(28, 43)
    parity           = rng.choice([0,1,2,3,4], n, p=[0.40,0.30,0.18,0.08,0.04])
    prev_csection    = np.where(rng.random(n) < 0.15,
                                rng.choice([1,2,3], n, p=[0.6,0.3,0.1]), 0)
    # 0=cephalic(80%), 1=breech(15%), 2=transverse(5%)
    presentation     = rng.choice([0,1,2], n, p=[0.80,0.15,0.05])
    labor_duration   = rng.exponential(8, n).clip(0, 48)
    cervical_dilation= rng.uniform(0, 10, n)
    fetal_weight_est = rng.normal(3.2, 0.5, n).clip(0.8, 5.5)
    maternal_height  = rng.normal(161, 7, n).clip(135, 190)
    amniotic_fluid   = rng.normal(13, 4, n).clip(1, 35)

    # ── Build logistic score for realistic C-section probability ──
    score = (
        2.0 * (prev_csection > 0).astype(float) +
        3.0 * (presentation == 1).astype(float) +
        4.0 * (presentation == 2).astype(float) +
        1.5 * (fetal_weight_est >= 4.0).astype(float) +
        1.2 * (labor_duration > 18).astype(float) +
        1.0 * (systolic_bp > 140).astype(float) +
        0.8 * (bmi >= 35).astype(float) +
        0.6 * (age > 38).astype(float) +
        0.5 * (amniotic_fluid < 5).astype(float) +
        rng.normal(0, 0.5, n)  # noise
    )
    prob_cs  = 1 / (1 + np.exp(-score + 1.5))  # logistic
    outcome  = (rng.random(n) < prob_cs).astype(int)

    df = pd.DataFrame({
        'Age': age, 'BMI': bmi.round(1),
        'SystolicBP': systolic_bp, 'DiastolicBP': diastolic_bp,
        'GestationalAge': gestational_age.round(1),
        'Parity': parity, 'PreviousCSections': prev_csection,
        'FetalPresentation': presentation,
        'LaborDuration': labor_duration.round(1),
        'CervicalDilation': cervical_dilation.round(1),
        'EstFetalWeight': fetal_weight_est.round(2),
        'MaternalHeight': maternal_height.round(0),
        'AmnioticFluidIndex': amniotic_fluid.round(1),
        'Outcome': outcome
    })
    return df

df = generate_csection_dataset(N, SEED)
print(f'Dataset shape: {df.shape}')
print(f'C-section rate: {df["Outcome"].mean()*100:.1f}%')
df.head()

## 3. EDA

In [ ]:
print('=== Class Distribution ===')
print(df['Outcome'].value_counts())

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
num_cols = ['Age','BMI','SystolicBP','GestationalAge',
            'LaborDuration','CervicalDilation','EstFetalWeight','AmnioticFluidIndex']
for i, col in enumerate(num_cols):
    df.groupby('Outcome')[col].hist(ax=axes[i], bins=20, alpha=0.7,
                                    color=['#10B981','#EC4899'], label=['Normal','C-Section'])
    axes[i].set_title(col)
    axes[i].legend()
plt.suptitle('Feature Distributions by Delivery Outcome', fontsize=14, color='#9D174D')
plt.tight_layout()
plt.show()

## 4. Feature Engineering (35 Features)

In [ ]:
def engineer_csection_features_df(df):
    """Matches engineer_csection_features() in app.py exactly."""
    d = df.copy()

    # === Raw inputs (13) ===
    # Already present: Age, BMI, SystolicBP, DiastolicBP, GestationalAge,
    # Parity, PreviousCSections, FetalPresentation, LaborDuration,
    # CervicalDilation, EstFetalWeight, MaternalHeight, AmnioticFluidIndex

    # === Clinical flags (12) ===
    d['PrevCSectionFlag']    = (d['PreviousCSections'] > 0).astype(int)
    d['MultiplePrevCS']      = (d['PreviousCSections'] >= 2).astype(int)
    d['BreechPresentation']  = (d['FetalPresentation'] == 1).astype(int)
    d['AbnormalPresent']     = (d['FetalPresentation'] > 0).astype(int)
    d['Macrosomia']          = (d['EstFetalWeight'] >= 4.0).astype(int)
    d['ObeseFlag']           = (d['BMI'] >= 30).astype(int)
    d['ElderlyMother']       = (d['Age'] > 35).astype(int)
    d['PostTerm']            = (d['GestationalAge'] > 41).astype(int)
    d['Preterm']             = (d['GestationalAge'] < 37).astype(int)
    d['ProlongedLabor']      = (d['LaborDuration'] > 18).astype(int)
    d['NulliparousFlag']     = (d['Parity'] == 0).astype(int)
    d['HypertensionFlag']    = (d['SystolicBP'] > 140).astype(int)
    d['OligohydramniosFlag'] = (d['AmnioticFluidIndex'] < 5).astype(int)
    d['PolyhydramniosFlag']  = (d['AmnioticFluidIndex'] > 24).astype(int)

    # === Derived features (7) ===
    d['PulsePressure']       = d['SystolicBP'] - d['DiastolicBP']
    d['MAP']                 = d['DiastolicBP'] + (d['SystolicBP'] - d['DiastolicBP']) / 3
    d['BMI_EFW_Ratio']       = d['BMI'] / (d['EstFetalWeight'] + 1e-6)
    d['Pelvis_EFW_Ratio']    = d['MaternalHeight'] / (d['EstFetalWeight'] * 10 + 1e-6)
    d['LaborProgress']       = d['CervicalDilation'] / (d['LaborDuration'] + 1e-6)
    d['AgeParityRatio']      = d['Age'] / (d['Parity'] + 1)
    d['GestAge_EFW']         = d['GestationalAge'] * d['EstFetalWeight']

    # === Risk score (1) ===
    d['CSectionRiskScore']   = (
        d['PrevCSectionFlag'] + d['BreechPresentation'] +
        d['Macrosomia'] + d['ProlongedLabor'] +
        d['ElderlyMother'] + d['HypertensionFlag']
    )

    return d


df_feat = engineer_csection_features_df(df)

FEATURE_COLS = [
    # Raw
    'Age', 'BMI', 'SystolicBP', 'DiastolicBP', 'GestationalAge',
    'Parity', 'PreviousCSections', 'FetalPresentation',
    'LaborDuration', 'CervicalDilation', 'EstFetalWeight',
    'MaternalHeight', 'AmnioticFluidIndex',
    # Flags
    'PrevCSectionFlag', 'MultiplePrevCS', 'BreechPresentation',
    'AbnormalPresent', 'Macrosomia', 'ObeseFlag', 'ElderlyMother',
    'PostTerm', 'Preterm', 'ProlongedLabor', 'NulliparousFlag',
    'HypertensionFlag', 'OligohydramniosFlag', 'PolyhydramniosFlag',
    # Derived
    'PulsePressure', 'MAP', 'BMI_EFW_Ratio', 'Pelvis_EFW_Ratio',
    'LaborProgress', 'AgeParityRatio', 'GestAge_EFW',
    # Score
    'CSectionRiskScore'
]
TARGET_COL = 'Outcome'

X = df_feat[FEATURE_COLS].values
y = df_feat[TARGET_COL].values

print(f'Feature matrix: {X.shape}  ({len(FEATURE_COLS)} features)')
print(f'Feature list:')
for i, f in enumerate(FEATURE_COLS, 1):
    print(f'  {i:2d}. {f}')

## 5. Train/Test Split + Balancing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

smt = SMOTETomek(random_state=SEED)
X_train_bal, y_train_bal = smt.fit_resample(X_train_sc, y_train)

print(f'Train before: {np.bincount(y_train)}')
print(f'Train after : {np.bincount(y_train_bal)}')
print(f'Test        : {np.bincount(y_test)}')

## 6. Baseline Model Comparison

In [ ]:
baseline_models = {
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=SEED),
    'LightGBM':            lgb.LGBMClassifier(n_estimators=200, random_state=SEED, verbose=-1),
    'XGBoost':             xgb.XGBClassifier(n_estimators=200, random_state=SEED,
                                              use_label_encoder=False, eval_metric='logloss'),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=SEED),
}

results = []
for name, model in baseline_models.items():
    model.fit(X_train_bal, y_train_bal)
    y_pred = model.predict(X_test_sc)
    acc = accuracy_score(y_test, y_pred) * 100
    f1  = f1_score(y_test, y_pred, average='macro') * 100
    auc = roc_auc_score(y_test, model.predict_proba(X_test_sc)[:, 1]) * 100
    results.append({'Model': name, 'Accuracy': round(acc,2), 'Macro F1': round(f1,2), 'ROC-AUC': round(auc,2)})
    print(f'{name:<22} Acc={acc:.2f}%  F1={f1:.2f}%  AUC={auc:.2f}%')

pd.DataFrame(results).sort_values('Macro F1', ascending=False)

## 7. Stacking Ensemble

In [ ]:
estimators = [
    ('lgb', lgb.LGBMClassifier(n_estimators=200, random_state=SEED, verbose=-1)),
    ('xgb', xgb.XGBClassifier(n_estimators=200, random_state=SEED,
                                use_label_encoder=False, eval_metric='logloss')),
    ('rf',  RandomForestClassifier(n_estimators=200, random_state=SEED)),
]
stacking_cs = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=500, random_state=SEED),
    cv=5, passthrough=False
)
stacking_cs.fit(X_train_bal, y_train_bal)
y_pred_s = stacking_cs.predict(X_test_sc)
acc_s = accuracy_score(y_test, y_pred_s) * 100
f1_s  = f1_score(y_test, y_pred_s, average='macro') * 100
print(f'Stacking Ensemble  Acc={acc_s:.2f}%  Macro-F1={f1_s:.2f}%')

## 8. Optuna Tuning (LightGBM — tune if LightGBM is best)

In [ ]:
def objective_cs(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 700),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 150),
        'max_depth':         trial.suggest_int('max_depth', 3, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 60),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 2.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 2.0, log=True),
        'random_state': SEED, 'verbose': -1
    }
    model = lgb.LGBMClassifier(**params)
    cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    return cross_val_score(model, X_train_bal, y_train_bal, cv=cv, scoring='f1_macro').mean()

study_cs = optuna.create_study(direction='maximize')
study_cs.optimize(objective_cs, n_trials=100, show_progress_bar=True)

print(f'\nBest CV F1: {study_cs.best_value*100:.2f}%')
print(f'Best params: {study_cs.best_params}')

## 9. Final Model & Evaluation

In [ ]:
best_p = study_cs.best_params
best_p.update({'random_state': SEED, 'verbose': -1})

cs_model = lgb.LGBMClassifier(**best_p)
cs_model.fit(X_train_bal, y_train_bal)

y_pred_cs = cs_model.predict(X_test_sc)
y_prob_cs = cs_model.predict_proba(X_test_sc)[:, 1]

print('=== C-SECTION MODEL — TEST RESULTS ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_cs)*100:.2f}%')
print(f'Macro F1 : {f1_score(y_test, y_pred_cs, average="macro")*100:.2f}%')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob_cs)*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_cs,
      target_names=['Normal Delivery', 'C-Section']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_cs)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu', ax=axes[0],
            xticklabels=['Normal', 'C-Section'],
            yticklabels=['Normal', 'C-Section'])
axes[0].set_title('Confusion Matrix', fontsize=13, color='#9D174D')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Feature importance
fi = pd.Series(cs_model.feature_importances_, index=FEATURE_COLS).nlargest(15)
fi.plot(kind='barh', ax=axes[1], color='#EC4899')
axes[1].set_title('Top 15 Feature Importances', fontsize=13, color='#9D174D')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 10. 10-Fold Cross-Validation

In [ ]:
cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(cs_model, X_train_bal, y_train_bal, cv=cv10, scoring='accuracy')
print(f'10-Fold CV Accuracy: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')

## 11. SHAP Explainability

In [ ]:
import shap

explainer_cs  = shap.TreeExplainer(cs_model)
shap_values_cs = explainer_cs.shap_values(X_test_sc)

sv = shap_values_cs[1] if isinstance(shap_values_cs, list) else shap_values_cs
shap.summary_plot(sv, X_test_sc, feature_names=FEATURE_COLS, show=True)

## 12. Save Model

In [ ]:
pickle.dump(cs_model,    open('model/csection_model.sav', 'wb'))
pickle.dump(scaler,      open('model/csection_scaler.sav', 'wb'))
pickle.dump(FEATURE_COLS, open('model/csection_feature_names.sav', 'wb'))

print('✅ Saved:')
print('   model/csection_model.sav')
print('   model/csection_scaler.sav')
print('   model/csection_feature_names.sav')

## 13. app.py Integration Snippet

In [ ]:
snippet = '''
# ── In app.py load_models() ────────────────────────────────────────
cs_model  = pickle.load(open("model/csection_model.sav",  "rb"))
cs_scaler = pickle.load(open("model/csection_scaler.sav", "rb"))

# ── Replace predict_csection_rule_based() ─────────────────────────
def predict_csection_model(age, bmi, systolic_bp, diastolic_bp,
                            gestational_age, parity, prev_csection,
                            fetal_presentation, labor_duration,
                            cervical_dilation, fetal_weight_est,
                            maternal_height, amniotic_fluid):
    features = engineer_csection_features(
        age, bmi, systolic_bp, diastolic_bp, gestational_age,
        parity, prev_csection, fetal_presentation, labor_duration,
        cervical_dilation, fetal_weight_est, maternal_height, amniotic_fluid
    )
    features_sc = cs_scaler.transform(features)
    prediction  = cs_model.predict(features_sc)[0]
    probs       = cs_model.predict_proba(features_sc)[0]
    confidence  = max(probs) * 100
    # Map binary → 3-level display
    risk_level = 0 if prediction == 0 else (1 if probs[1] < 0.70 else 2)
    return risk_level, confidence, []
'''
print(snippet)

---
### ✅ C-Section Model Training Complete
Now run notebook `03_fetal_birth_weight_model_training.ipynb`.